In [1]:
import torch
import torch.nn as nn
import networkx as nx
import numpy as np

# 1. Define the RNN Architecture
class SentimentRNN(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim):
        super(SentimentRNN, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        # Use SimpleRNN or LSTM for sequence processing
        self.rnn = nn.RNN(embed_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x, h0=None):
        x = self.embedding(x)
        out, h_n = self.rnn(x, h0)
        # We use the final hidden state to predict sentiment
        prediction = self.fc(h_n.squeeze(0))
        return self.sigmoid(prediction), h_n

# 2. Setup the Network Graph
G = nx.Graph()
# Example nodes with text content
node_data = {
    0: "This product is amazing!", # Positive
    1: "Terrible experience.",     # Negative
    2: "Really loved it."          # Positive
}
G.add_edges_from([(0, 1), (1, 2)]) # Connection between nodes

# 3. Preprocessing (Tokenization & Padding)
# Simplistic mapping for demonstration
vocab = {"<PAD>": 0, "this": 1, "product": 2, "is": 3, "amazing": 4, "terrible": 5, "experience": 6, "really": 7, "loved": 8, "it": 9}
def preprocess(text, max_len=5):
    tokens = [vocab.get(w, 0) for w in text.lower().replace("!", "").replace(".", "").split()]
    return tokens[:max_len] + [0] * (max_len - len(tokens))

# 4. Sentiment Inference with Network Influence
model = SentimentRNN(len(vocab), 16, 32)
node_hiddens = {}

# Pass 1: Get individual node sentiment hidden states
for node_id, text in node_data.items():
    tokens = torch.LongTensor([preprocess(text)])
    _, h_n = model(tokens)
    node_hiddens[node_id] = h_n.detach()

# Pass 2: Refine sentiment based on neighbors
for node_id in G.nodes():
    neighbors = list(G.neighbors(node_id))
    if neighbors:
        # Average neighbor hidden states to simulate social influence
        neighbor_context = torch.mean(torch.stack([node_hiddens[n] for n in neighbors]), dim=0)

        # Re-run prediction using neighbor context as initial hidden state
        tokens = torch.LongTensor([preprocess(node_data[node_id])])
        prediction, _ = model(tokens, neighbor_context)

        sentiment = "Positive" if prediction.item() > 0.5 else "Negative"
        print(f"Node {node_id} Sentiment: {sentiment} ({prediction.item():.2f})")


Node 0 Sentiment: Positive (0.65)
Node 1 Sentiment: Positive (0.67)
Node 2 Sentiment: Positive (0.68)
